# Load neceecities

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import numpy as np
import os
import json
import time
import gc
import torch.nn.functional as F
import math
import math
import pickle as pkl
import functools
from transformers.models.llama.modeling_llama import LlamaAttention
wd = os.getcwd()
import random
from scipy.stats import zscore

h:\conda_env\glm3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
main_prompt = "answer the question based on the following text,keep the your response short and simple, do not quote the original text."
base_prompt = '''{main_prompt},
question:
{question}
context:
{context}
'''
with open('stqa_350.json', 'r') as f:
    qa = json.load(f)


In [7]:
def find_start_end_index_input_llama(model_inputs):
    input_start = model_inputs.input_ids[0].tolist().index(tokenizer.convert_tokens_to_ids('system')) + 23
    input_end = model_inputs.input_ids[0].tolist().index(tokenizer.convert_tokens_to_ids('assistant')) -3
    print(f'input_start: {input_start}, input_end: {input_end}')
    return input_start, input_end

def find_context_index(model_inputs,tokenizer,context_ids=None,context_marker = 'context'):
    context_index = None
    if context_ids == None:
        context_ids = tokenizer.convert_tokens_to_ids(context_marker)
    for i,t in enumerate(model_inputs.input_ids[0].tolist()):
        if 'context' in tokenizer.decode([t]):
            context_index = i
            break
    return context_index

import numpy as np
from scipy.stats import zscore # Make sure scipy is installed and this can be imported

def generate_text_with_scores_html(tensor, text, output_path, normalize=True, method='min-max', window_size=10, msg=True):
    """
    Generates a beautified HTML file based on the input vector and text, where each character
    is displayed with its corresponding score above it, and the color is adjusted according to the score.
    The final HTML file is saved to the specified path.

    Args:
    tensor (numpy.ndarray): A vector of shape (1, N), where N is the length of the text.
    text (str): The text string, matching the length of the vector.
    output_path (str): Path to save the output HTML file.
    normalize (bool): Whether to normalize the scores, default is True.
    method (str): Normalization method; supports 'min-max' (default), 'mean', 
                  'moving-average', 'z-score', 'max_average', or 'z-score_exclude_max'.
    window_size (int): Window size for moving average, default is 10; used only 
                       when 'moving-average' normalization is selected.
    msg (bool): Whether to print save information, default is True.

    Returns:
    None
    """
    
    assert tensor.shape[1] == len(text), "Mismatch between tensor and text length!"
    
    scores_for_display_and_boxing = tensor[0].astype(float) # Work with a copy as float
    epsilon = 1e-9 # For safe division

    if normalize:
        s_min = scores_for_display_and_boxing.min()
        s_max = scores_for_display_and_boxing.max()
        
        if method == 'min-max':
            delta = s_max - s_min
            if delta < epsilon: # Handles case where all scores are the same
                scores_for_display_and_boxing = np.full_like(scores_for_display_and_boxing, 0.0)
            else:
                scores_for_display_and_boxing = (scores_for_display_and_boxing - s_min) / delta
        elif method == 'mean':
            mean_value = scores_for_display_and_boxing.mean()
            scores_for_display_and_boxing = scores_for_display_and_boxing - mean_value
        elif method == 'moving-average':
            if len(scores_for_display_and_boxing) < window_size:
                 # Fallback for short sequences or provide a warning/error
                mean_value = scores_for_display_and_boxing.mean()
            else:
                mean_value = np.convolve(scores_for_display_and_boxing, np.ones(window_size) / window_size, mode='same')
            scores_for_display_and_boxing = scores_for_display_and_boxing - mean_value
        elif method == 'z-score':
            # zscore function expects 2D input for axis=1, or 1D
            if scores_for_display_and_boxing.ndim == 1:
                 scores_for_display_and_boxing = zscore(scores_for_display_and_boxing, ddof=0, nan_policy="omit")
            else: # Should be (1,N) already from tensor[0] but as 1D array
                 scores_for_display_and_boxing = zscore(scores_for_display_and_boxing.reshape(1, -1), axis=1, ddof=0, nan_policy="omit")[0]
            scores_for_display_and_boxing = np.nan_to_num(scores_for_display_and_boxing) # Handle potential NaNs if std is 0
        elif method == 'max_average':
            # Assumes scores are typically positive; scales by max.
            # If max is zero or negative, behavior might be undesirable without further context.
            # This will map values to be <= 1.
            current_max = s_max
            if abs(current_max) < epsilon : # Avoid division by zero or near-zero if max is very small
                if current_max >= 0: # if max is 0 or positive epsilon
                     scores_for_display_and_boxing = np.zeros_like(scores_for_display_and_boxing)
                # else: if max is negative epsilon, dividing by it would flip signs and magnify.
                # This case might need specific handling based on expected score nature.
                # For now, if max is negative, this normalization might not be ideal.
                # Let's assume for 'max_average', we expect positive scores or scale by abs(max)
            elif current_max < 0: # If max score is negative, dividing makes them more positive.
                                  # This might be unexpected. Consider absolute max or specific logic.
                                  # For now, proceed with s_max.
                scores_for_display_and_boxing = scores_for_display_and_boxing / (current_max - epsilon) # ensure denominator non-zero
            else:
                scores_for_display_and_boxing = scores_for_display_and_boxing / (current_max + epsilon)

        elif method == 'z-score_exclude_max':
            if len(scores_for_display_and_boxing) <= 1:
                 scores_for_display_and_boxing = np.zeros_like(scores_for_display_and_boxing)
            else:
                max_value_idx = np.argmax(scores_for_display_and_boxing)
                temp_scores = np.delete(scores_for_display_and_boxing, max_value_idx)
                if len(temp_scores) == 0: # Only one element initially
                     mean_value = 0
                     std_value = 1.0 # Avoid division by zero if only one element was present
                else:
                     mean_value = temp_scores.mean()
                     std_value = temp_scores.std()
                
                std_value = std_value + epsilon # Add epsilon to std_dev
                scores_for_display_and_boxing = (scores_for_display_and_boxing - mean_value) / std_value
        else:
            raise ValueError("Unknown normalization method. Please use 'min-max', 'mean', 'moving-average', 'z-score', 'max_average', or 'z-score_exclude_max'.")

    # Normalize scores for color mapping to a [0, 1] range
    # This ensures the full color spectrum is used for the current set of scores.
    s_min_color = scores_for_display_and_boxing.min()
    s_max_color = scores_for_display_and_boxing.max()
    delta_color = s_max_color - s_min_color
    
    if delta_color < epsilon: # All display scores are effectively the same
        scores_for_color_mapping = np.full_like(scores_for_display_and_boxing, 0.5) # Neutral color (purple)
    else:
        scores_for_color_mapping = (scores_for_display_and_boxing - s_min_color) / delta_color

    # Create color mapping function (Blue for low, Red for high, Purple for mid)
    def score_to_color(score_0_to_1):
        score_0_to_1 = np.clip(score_0_to_1, 0.0, 1.0) # Ensure score is strictly within [0,1]
        r = int(255 * score_0_to_1)
        b = 255 - r
        # Ensure text is readable on colored background by choosing black or white text
        # This simple luminance check might need refinement for specific color schemes
        luminance = 0.299 * r + 0.587 * 0 + 0.114 * b # Green is 0 in this color scheme
        # Text color for the score and character itself.
        # The original function set the text color using this.
        return f'rgb({r}, 0, {b})'

    chars_per_line = 50 # You can adjust this
    num_lines = len(text) // chars_per_line + (1 if len(text) % chars_per_line else 0)

    html_content = """
<html>
<head>
    <title>Text with Scores</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f8f9fa;
            color: #333;
            line-height: 1.6;
        }
        .container {
            max-width: 1200px;
            margin: 20px auto;
            background: #ffffff;
            padding: 25px;
            border-radius: 8px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.1);
        }
        h2 {
            text-align: center;
            color: #007bff; /* A nice blue for the title */
            margin-bottom: 20px;
        }
        .description, .legend {
            font-size: 0.95em;
            color: #555;
            background-color: #e9ecef;
            padding: 15px;
            border-radius: 5px;
            margin-bottom: 20px;
        }
        .legend ul {
            padding-left: 20px;
            margin: 0;
        }
        .line-container {
            display: flex;
            align-items: flex-start; /* Align line number with the top of char units */
            margin-bottom: 12px; /* Space between lines of text */
            flex-wrap: nowrap; /* Prevent line number and content from wrapping against each other */
        }
        .line-number {
            min-width: 40px; /* Space for line numbers */
            font-size: 0.8em;
            color: #6c757d; /* Bootstrap's secondary text color */
            padding-right: 10px;
            padding-top: 0.2em; /* Align with score text vertically */
            text-align: right;
            font-family: "Courier New", Courier, monospace;
        }
        .line-content {
            display: flex;
            flex-wrap: wrap; /* Allow character units to wrap if line is too long for container */
            font-family: "Courier New", Courier, monospace;
            flex-grow: 1;
        }
        .char-unit {
            display: inline-flex;
            flex-direction: column;
            align-items: center;
            justify-content: center; /* Center content vertically */
            margin-right: 4px; /* Space between character units */
            margin-bottom: 4px; /* Space if units wrap */
            padding: 3px 5px;
            border-radius: 4px;
            min-width: 2.2em; /* Ensure some consistency in width */
            min-height: 3.5em; /* Ensure some consistency in height */
            text-align: center;
            transition: transform 0.2s ease-out, box-shadow 0.2s ease-out; /* Smooth hover effect */
        }
        .char-unit:hover {
            transform: translateY(-2px); /* Slight lift on hover */
            box-shadow: 0 2px 5px rgba(0,0,0,0.1); /* Shadow on hover */
        }
        .score-value {
            font-size: 0.65em; /* Smaller font for score */
            margin-bottom: 3px;
            font-weight: normal;
        }
        .char-value {
            font-size: 1.1em; /* Larger font for character */
            font-weight: bold;
        }
        .char-unit.boxed {
            background-color: rgba(40, 167, 69, 0.1); /* Light green background for positive scores */
            border: 1px solid rgba(40, 167, 69, 0.3); /* Greenish border */
        }
        .color-swatch {
            display: inline-block;
            width: 15px;
            height: 15px;
            border: 1px solid #ccc;
            margin-right: 5px;
            vertical-align: middle;
        }
    </style>
</head>
<body>
    <div class="container">
        <h2>Text Visualization with Scores</h2>
        <div class="description">
            <p>Each character from the input text is displayed below its corresponding numerical score. The color of the text (character and score) visually represents the score's magnitude within the current dataset's range:</p>
            <ul>
                <li><span class="color-swatch" style="background-color: rgb(255,0,0);"></span> Scores towards the <strong>maximum</strong> value in the dataset are colored <strong>red</strong>.</li>
                <li><span class="color-swatch" style="background-color: rgb(127,0,128);"></span> Scores around the <strong>middle</strong> of the range are colored <strong>purple</strong>.</li>
                <li><span class="color-swatch" style="background-color: rgb(0,0,255);"></span> Scores towards the <strong>minimum</strong> value in the dataset are colored <strong>blue</strong>.</li>
            </ul>
            <p>The displayed numerical score is the value after the selected normalization method (or the raw score if normalization is off). Characters whose displayed scores are <strong>greater than zero</strong> are highlighted with a light green background and border.</p>
        </div>
"""

    for line_idx in range(num_lines):
        html_content += f"<div class='line-container'>\n"
        html_content += f"  <span class='line-number'>{line_idx + 1}</span>\n"
        html_content += f"  <div class='line-content'>\n"
        
        start_idx = line_idx * chars_per_line
        end_idx = start_idx + chars_per_line
        
        line_text_chars = text[start_idx:end_idx]
        line_display_scores = scores_for_display_and_boxing[start_idx:end_idx]
        line_color_scores = scores_for_color_mapping[start_idx:end_idx]

        for i, char_val in enumerate(line_text_chars):
            display_score = line_display_scores[i]
            color_score = line_color_scores[i]
            
            char_color = score_to_color(color_score)
            score_text_val = f"{display_score:.2f}"
            
            box_class = "boxed" if display_score > 0 else ""
            
            # Handle special HTML characters like '<', '>', '&'
            char_display = char_val.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace(" ", "&nbsp;")
            if char_display == "&nbsp;" and char_val != " ": # If original was not space, but became nbsp due to other replacements
                char_display = char_val # Revert if it wasn't a space

            html_content += (
                f"<div class='char-unit {box_class}'>"
                f"<span class='score-value' style='color:{char_color};'>{score_text_val}</span>"
                f"<span class='char-value' style='color:{char_color};'>{char_display}</span>"
                f"</div>"
            )

        html_content += "\n  </div>\n</div>\n"

    html_content += """
    </div>
</body>
</html>
"""

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    if msg:
        print(f"HTML file has been generated and saved to {output_path}")



# MACS

## Load Llama3.1-8B

In [ ]:
device = "cuda" # the device to load the model onto
model_id = 'meta-llama/Llama-3.1-8B' # the model ID or path
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
     attn_implementation="eager" , # Setting attn_implementation to "eager" is a MUST
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.eval()
model_patched = False

Loading checkpoint shards: 100%|██████████| 4/4 [00:24<00:00,  6.02s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


## Main MACS code

This Code is design to work with Llama3.1,to apply MACS on other models,you just need to change the default layer parameters and switch off the "excluede_special_tokens" since the special tokens are model specific
Strongly recommanded keep excluede_special_tokens on,Since the decoder-model (due to attention sink effect) will always put more attention to the first BOS token

In [9]:
def generate_text(model, tokenizer, device, prompt,heatmap_folder_name, max_new_tokens=256, model_id='', answers=None,
                  verbose_gen = False, masking_gen = False, mask_type = 'max',
                    excluede_special_tokens = True, only_context = True,verbose = False,
                    patched_model = False,
                    output_step_heatmap = False,
                    output_overall_heatmap = False,
                    output_raw_scores = False,
                    layer = 28, # for llama3, if you are using other models, please check the number of layers
                    ):
    wd = os.getcwd()
    step_heatmap_folder_map_path = os.path.join(wd, 'heatmap', heatmap_folder_name)
    overall_heatmap_folder_map_path = os.path.join(wd, 'heatmap', heatmap_folder_name, 'overall')
    if not os.path.exists(step_heatmap_folder_map_path):
        os.makedirs(step_heatmap_folder_map_path, exist_ok=True)
    if not os.path.exists(overall_heatmap_folder_map_path):
        os.makedirs(overall_heatmap_folder_map_path, exist_ok=True)
    """
    Generates text using a pre-trained language model and computes attention scores and other statistics during generation.
    
    Args:
        model: Pre-trained language model, e.g., GPT-2.
        tokenizer: Corresponding tokenizer object.
        device: Device to run the model on, such as 'cuda' or 'cpu'.
        prompt: Initial input text.
        max_new_tokens: Maximum number of tokens to generate, default is 256.
        model_id: Model identifier string to distinguish between models (e.g., 'qwen' or 'llama').
        answers: (Optional) Reference answers, used for printing.

    Returns:
        A dictionary containing the generation result, statistics, and attention scores.
    """

    # Clean up environment
    wd = os.getcwd()
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Configuration parameters
    excluede_special_tokens = True  # Remove special tokens, enabled by default
    only_context = True
    masking_gen = masking_gen
    verbose = verbose
    mask_type = mask_type
    verbose_gen = verbose_gen
   
    # Used to store information for each generation step
    gen_step_token = []

    # Model-related parameters
 
  

    alpha = 0.8
    attention_scores_by_step = []

    cumulative_logits = 0
    logits_mean = 0

    # Initially, no masked_positions
    masked_positions = None

    eos_token_id = tokenizer.eos_token_id

    # Prepare initial input
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    input_length = len(model_inputs['input_ids'][0])
    masking_portion = int(0.15 * input_length)
    
    # Choose different handling based on model_id
    input_start_index, input_end_index = find_start_end_index_input_llama(model_inputs)
    context_start = find_context_index(model_inputs,tokenizer)+1
    input_start_index = context_start

    # Get token ids and token strings for the input sequence
    if excluede_special_tokens:
        seq_ids = (model_inputs.input_ids[0].tolist())[input_start_index:input_end_index]
    else:
        seq_ids = model_inputs.input_ids[0].tolist()
    seq_tokens = [tokenizer.decode([token]) for token in seq_ids]
    
    if answers is not None:
        print('GOLD RESPONSE:\n', answers)
    
    initial_input_ids = model_inputs.input_ids
    current_input_ids = initial_input_ids.clone()
    gen_count = 0

    total_neg_log_prob = 0.0
    token_count = 0

    start_time = time.time()
    end_time = None

    # Generation loop
    while current_input_ids.shape[1] - initial_input_ids.shape[1] < max_new_tokens:
        outputs_row = []
        seq_len = current_input_ids.size(1)
        # Create causal mask
        causal_mask = torch.tril(torch.ones((seq_len, seq_len), device=device))
        attention_mask = causal_mask.unsqueeze(0)
        if masked_positions:
            if verbose:
                print('Masking')
            for p in masked_positions:
                if p < seq_len:
                    attention_mask[:, :, p] = 0
            outputs = model(input_ids=current_input_ids, attention_mask=attention_mask,
                            return_dict=True, output_attentions=True)
        else:
            outputs = model(input_ids=current_input_ids, return_dict=True, output_attentions=True)
        
        # Get logits at the last position
        next_token_logits = outputs.logits[:, -1, :]
        max_logit = torch.max(next_token_logits).detach().to(torch.float32).cpu().numpy()
        cumulative_logits += max_logit
        
        # Greedy decoding to get the next token
        next_token_id = torch.argmax(next_token_logits, dim=-1)
        log_probs = torch.log_softmax(next_token_logits, dim=-1)
        selected_log_prob = log_probs[0, next_token_id.item()]
        total_neg_log_prob += -selected_log_prob.item()
        token_count += 1

        next_token_token = tokenizer.decode(next_token_id).replace('\n', '\\n')
        current_gen_token = tokenizer.decode(current_input_ids[:, -1]).replace('\n', '\\n') if gen_count > 0 else 'First token, no previous token'
        # Save the generation info for this step as a dictionary
        step_data = {'step': gen_count, 'gen_token': next_token_token, 'pre_token': current_gen_token}
        gen_step_token.append(step_data)
        
        if verbose_gen:
            print(f"Gen step:{gen_count} : {next_token_token}")
            if gen_count == 0:
                print("Use this token to gen : First token, no previous token")
            else:
                print(f"Use this token to gen : {current_gen_token}")
            print('===============================================================')
        gen_count += 1
        
        # Check if EOS token is generated
        if next_token_id.item() == eos_token_id:
            end_time = time.time()
            logits_mean = cumulative_logits / gen_count
            break
        
        if verbose:
            print(outputs['attentions'][0].shape)

        # Process attention information (assuming number of attentions returned is not less than `layer`)
        temp = [outputs['attentions'][i].detach().cpu() for i in range(layer)]
        temp = torch.stack(temp).squeeze()
        #print(temp.shape)
        row = temp[:, :, -1, :]
        del temp
        input_row = row[:, :, :input_length]
        #print(input_row.shape)
        if gen_count > 0:
            outputs_row = row[:, :, input_length:]
            if excluede_special_tokens:
                outputs_row = outputs_row.sum(axis=-1) / (input_end_index - input_start_index)
            else:
                outputs_row = outputs_row.sum(axis=-1) / input_length
            input_row = input_row + outputs_row[:, :, np.newaxis]
            
        if excluede_special_tokens and only_context:
            input_row = input_row[:, :, input_start_index:input_end_index]
        input_row = input_row.to(torch.float32).numpy()
        res_att_mat = input_row.max(axis=1)[:, np.newaxis, :]
        bias_vector = np.ones(input_row.shape[-1])[None, None, :]
        #print(res_att_mat.shape)
        res_att_mat = alpha * res_att_mat + (1 - alpha) * bias_vector
        joint_att = np.zeros(res_att_mat.shape)
        joint_att[0] = res_att_mat[0]
        for i in np.arange(1, layer):
            joint_att[i] = res_att_mat[i] * joint_att[i-1]
        
        joint_att_heads_mean = joint_att
        
        last_layer = joint_att_heads_mean[-1, :, :]
        if output_step_heatmap:
            if ':' in seq_tokens[0] or '\n' in seq_tokens[0] : # a quike and dirty fix an...somehow the find_context_index function is returning the wrong index
                generate_text_with_scores_html(last_layer[:,1:], seq_tokens[1:], normalize=True,
                                           method='z-score', output_path=f'{step_heatmap_folder_map_path}/genstep_{gen_count}_gentoken_{next_token_token}.html', msg=False)
        if output_raw_scores:
            z_s = last_layer
        else:
            z_s = zscore(last_layer, axis=1, ddof=0, nan_policy="omit")[0]
        max_5_zscore = np.argsort(z_s)[-masking_portion:][::-1]
        max_5_zscore_token = [seq_tokens[i] for i in max_5_zscore]
        min_5_zscore = np.argsort(z_s)[:masking_portion]
        min_5_zscore_token = [seq_tokens[i] for i in min_5_zscore]
        # Store attention and z-score info in the current generation step dictionary
        gen_step_token[gen_count-1]['max_5_zscore'] = max_5_zscore.tolist()
        gen_step_token[gen_count-1]['min_5_zscore'] = min_5_zscore.tolist()
        gen_step_token[gen_count-1]['max_5_zscore_token'] = max_5_zscore_token
        gen_step_token[gen_count-1]['min_5_zscore_token'] = min_5_zscore_token
        gen_step_token[gen_count-1]['z_score'] = z_s
        gen_step_token[gen_count-1]['attention'] = joint_att_heads_mean
        if masking_gen and mask_type == 'max':
            masked_positions = max_5_zscore.tolist()
        elif masking_gen and mask_type == 'min':
            masked_positions = min_5_zscore.tolist()
        if verbose:
            print(f'max_5_zscore_token: {max_5_zscore_token}')
        
        # Append the new token to the current input sequence
        current_input_ids = torch.cat([current_input_ids, next_token_id.unsqueeze(1)], dim=1)
    
    # If exited without generating EOS, supplement end_time
    if end_time is None:
        end_time = time.time()
    
    # Aggregate attention info for each step
    attention_scores_by_step = []
    for step_data in gen_step_token[:-1]:
        # The 'attention' key in each dictionary stores the attention of the current step
        attention_scores_by_step.append(step_data['attention'])
    last_layer_step_attentkion = np.array([att[-1, :, :] for att in attention_scores_by_step])
    last_layer_max_step_attention = last_layer_step_attentkion.max(axis=0)
    if output_overall_heatmap:
        #print(last_layer_max_step_attention.shape)
        #print(seq_tokens)
        if ':' in seq_tokens[0] or '\n' in seq_tokens[0] : # a quike and dirty fix an...somehow the find_context_index function is returning the wrong index
            generate_text_with_scores_html(last_layer_max_step_attention[:,1:], seq_tokens[1:], normalize=True,
                                    method='z-score', output_path=f'{overall_heatmap_folder_map_path}/gen_max_step.html', msg=False)
    
    if output_raw_scores:
        z_s_max_step = last_layer_max_step_attention
    else:
        z_s_max_step = zscore(last_layer_max_step_attention, axis=1, ddof=0, nan_policy="omit")[0]
    max_5_zscore_max_step = np.argsort(z_s_max_step)[-masking_portion:][::-1]
    max_5_zscore_token_max_step = [seq_tokens[i] for i in max_5_zscore_max_step]
    print(f'max_5_zscore_token_max_step: {max_5_zscore_token_max_step}')
    overall = {
        'z_s_max_step': z_s_max_step.tolist(),
        'max_5_zscore_max_step': max_5_zscore_max_step.tolist(),
        'max_5_zscore_token_max_step': max_5_zscore_token_max_step
    }
    
    # Decode the generated result and compute related statistics
    generated_ids = current_input_ids[:, initial_input_ids.shape[1]:]
    inference_time = end_time - start_time
    num_tokens = generated_ids.shape[1]
    tokens_per_second = num_tokens / inference_time if inference_time > 0 else 0
    vram_peak = torch.cuda.max_memory_allocated() / (1024 * 1024)  # In MB
    response = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    
    if token_count > 0:
        avg_neg_log_prob = total_neg_log_prob / token_count
        perplexity = math.exp(avg_neg_log_prob)
        print(f"Overall perplexity of the generation steps: {perplexity}")
    else:
        print("No token was generated, unable to compute perplexity.")
        perplexity = None
    
    inference_stats = {
        'inference_time': inference_time,
        'num_tokens': num_tokens,
        'num_tokens_per_second': tokens_per_second,
        'vram_peak': vram_peak,
        'mean_logits': logits_mean,
        'perplexity': perplexity
    }
    final_result = {
        'inf_stats': inference_stats,
        'gen_step_token': gen_step_token,
        'overall': overall,
        'response': response,
        'model_input': model_inputs,
        'input_tokens': seq_tokens,
    }
    print('Model response:\n', response)
    if output_step_heatmap or output_overall_heatmap:
        print('HTML files have been generated and saved to the online folder')
    return final_result




In [ ]:
q = 212
question = qa[q]['question']
context = qa[q]['context']
answers = qa[q]['answers']
context = """The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were  the people who in the 10th and 11th centuries gave their name to Normandy,  a region in France. They were descended from Norse (” Norman” comes from  ” Norseman” ) raiders and pirates from Denmark, Iceland and Norway who, under  their leader Rollo, agreed to swear fealty to King Charles III of West Francia.  Through generations of assimilation and mixing with the native Frankish and  Roman-Gaulish populations, their descendants would gradually merge with the  Carolingian-based cultures of West Francia. The distinct cultural and ethnic  identity of the Normans emerged initially in the first half of the 10th century,  and it continued to evolve over the succeeding centuries."""
question = 'In what country is Normandy located?'
answers = 'France'
heatmap_folder_name = f'stqa_350_{q}'
print('Question:', question)
print('Context:', context)
print('================================================')
result = generate_text(
    model,
    tokenizer,
    device,
    prompt=base_prompt.format(main_prompt=main_prompt, question=question, context=context),
    max_new_tokens=256,
    model_id=model_id,
    answers=answers, # this is just for printing and storing the gold response
    verbose_gen=False,
    masking_gen=False,
    mask_type='max',
    excluede_special_tokens=True,
    only_context=True,
    verbose=False,
    patched_model=model_patched,
    output_step_heatmap=True,
    output_overall_heatmap=True,
    output_raw_scores=False,
    heatmap_folder_name=heatmap_folder_name, # this is the folder name to save the html files
)

Question: In what country is Normandy located?
Context: The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were  the people who in the 10th and 11th centuries gave their name to Normandy,  a region in France. They were descended from Norse (” Norman” comes from  ” Norseman” ) raiders and pirates from Denmark, Iceland and Norway who, under  their leader Rollo, agreed to swear fealty to King Charles III of West Francia.  Through generations of assimilation and mixing with the native Frankish and  Roman-Gaulish populations, their descendants would gradually merge with the  Carolingian-based cultures of West Francia. The distinct cultural and ethnic  identity of the Normans emerged initially in the first half of the 10th century,  and it continued to evolve over the succeeding centuries.
input_start: 26, input_end: 235
GOLD RESPONSE:
 France


## Optional : Patch the Llama model (WIP)

In [ ]:
import functools
from transformers.models.llama.modeling_llama import LlamaAttention

#Optional:Monkey patching LlamaAttention so the output_attentions only returns the max-pooled last row of attention weights
#This will substantially reduce the memory footprint of the MACS

print('--- Warning:Experimental patching of LlamaAttention to reduce memory footprint ---')
print('--- WIP,Need to just the generate_text function,it currently only takes the full attention matrix ---')
_orig_attn_forward = LlamaAttention.forward

@functools.wraps(_orig_attn_forward)
def _patched_attn_forward(
    self, hidden_states, position_embeddings=None,
    attention_mask=None, past_key_value=None,
    cache_position=None, output_attentions=False, **kwargs
):
    outputs = _orig_attn_forward(
        self, hidden_states=hidden_states,
        position_embeddings=position_embeddings,
        attention_mask=attention_mask,
        past_key_value=past_key_value,
        cache_position=cache_position,
        output_attentions=output_attentions,
        **kwargs
    )
    if not output_attentions or len(outputs) < 2 or outputs[-1] is None:
        return outputs

    *rest, full_attn = outputs      # full_attn: [B,H,S,S]
    # 1) last row
    last_row = full_attn[:, :, -1, :]       # type: ignore # [B, H, S]
    # 2) head 维度做 max → [B, S]
    pooled = last_row.max(dim=1).values
    pooled = pooled.unsqueeze(1) # [B, 1, S]
    # if can unsqueeze to [B, 1, S] and then do a matmul with the last row
    return (*rest, pooled)

LlamaAttention.forward = _patched_attn_forward # type: ignore
model_patched = True

## VQA implementation (Warning,dirty code)

In [ ]:

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto",attn_implementation="eager"
) #Use 3B model for showcasing,if your GPU is powerful enough, you can use 7B one: Qwen/Qwen2.5-VL-7B-Instruct
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')


# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
min_pixels = 256*28*28
max_pixels = 1280*28*28
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)
think = False
only_context = False
context_start = None
def find_image_index (input,s_id = 151652,e_id = 151653):
    i_start = 0
    i_end = 0 
    if type(input) != list:
        input = input.tolist()
    for i in range (len(input)):
        if input[i] == s_id:
            i_start = i
        if input[i] == e_id:
            i_end = i
    return i_start+1,i_end

Loading checkpoint shards: 100%|██████████| 2/2 [00:35<00:00, 17.75s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [7]:

demo = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
im1 = 'https://hips.hearstapps.com/ghk.h-cdn.co/assets/17/30/pembroke-welsh-corgi.jpg?crop=1xw:0.9997114829774957xh;center,top&resize=980:*'
im2 = 'https://upload.wikimedia.org/wikipedia/commons/3/3b/BlkStdSchnauzer2.jpg'
#im3 = 'https://upload.wikimedia.org/wikipedia/commons/9/9c/Siberian_Husky_pho.jpg'
im3 = 'Siberian_Husky_pho.jpg'
im4 = 'https://images.theconversation.com/files/625056/original/file-20241010-19-5h51ab.jpg?ixlib=rb-4.1.0&q=45&auto=format&w=1000&fit=clip'

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": im3,
            },
            {"type": "text", "text": "What is in the image?"},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
#generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids = model.generate(**inputs, max_new_tokens=256,return_dict_in_generate=True,
    output_attentions = True,)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids.sequences)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


["The image shows a young child and a dog, likely a Siberian Husky, sitting together in a grassy outdoor setting. The child is wearing a red cap, a red and white jacket, and has a backpack on. The dog is wearing a harness and appears to be licking the child's face. The background includes some trees and a cloudy sky."]


In [8]:
input_start_index, input_end_index = find_image_index(inputs['input_ids'][0])
attentions_step1 = generated_ids['attentions'][0]
model_inputs = inputs

# Create a list to store the attention for the last token at each layer
last_token_attention_list = []

# assert generated_ids[0][0][end_index:end_index+1] == end_token
# print(tokenizer.decode(generated_ids[0][0][:input_length][-1]).replace('\n','\\n'))  # Should output '<|im_end|>' or '\n', to confirm the last token in the input sequence is '<|im_end|>' or '\n'

# Iterate over each layer's attention matrix in the tuple
for layer_attention in attentions_step1:
    # Select the attention to the last generated token and append to the list
    last_token_attention = layer_attention[:, :, -1, :].unsqueeze(2)
    last_token_attention_list.append(last_token_attention)

# You can convert this list to a tensor or keep it as a list
# last_token_attention_tensor = torch.stack(last_token_attention_list)

# Check the shape of the last token attention from the first layer
print(last_token_attention_list[0].shape)
print(len(last_token_attention_list))

# assert that the length of attention vector matches input length
# assert last_token_attention_list[0].shape[-1] == len(model_inputs['input_ids'][0])

print(last_token_attention_list[-1].squeeze()[-1].sum().item())
assert last_token_attention_list[-1].squeeze()[-1].sum().item() == 1.0  # Confirm this is row-wise attention weights summing to 1

# Convert generated_ids['attentions'] to a list
attentions_list = list(generated_ids['attentions'])

# Replace the tuple for generation step 0 with the new one
attentions_list[0] = tuple(last_token_attention_list)

# Convert the list back to tuple and assign to generated_ids['attentions']
generated_ids['attentions'] = tuple(attentions_list)

# Verify the replaced content
print(type(generated_ids['attentions']))  # Should be tuple
print(generated_ids['attentions'][0][0].shape)  # Should be [1, 28, 1, 1776]

import torch
import numpy as np

# Create a new list to store the converted tensors
# Suppose the input length is 66, which includes special tokens and system prompt; the actual prompt starts from position 54
converted_attentions = []
converted_attentions_output_self = []
converted_attentions_response2think = []
converted_attentions_think2input = []
converted_attentions_response2input = []

# input_start_index and input_end_index indicate the start and end positions of the input sequence
# output_start and output_end indicate the start and end positions of the generated sequence
# think_start and think_end indicate the COT (Chain-of-Thought) segment within the generated sequence
input_length = len(model_inputs['input_ids'][-1])
input_length_without_special_tokens = input_end_index - input_start_index

# if excluede_special_tokens and only_context:
    # input_length = input_length_without_special_tokens

for i in range(len(generated_ids['attentions'])):

    # For each generation step, concatenate the 28 attention layers (shape [batch_size=1, head=28, seq_len=1, seq_len=input+generated])
    # into one tensor of shape [layer=28, head=28, seq_len=1, seq_len=(input+generated)]
    step_attentions = torch.cat([layer for layer in generated_ids['attentions'][i]], dim=0)

    # Keep only the content from start_index to end_index
    # if step_attentions.shape[-1] >= input_length:

    step_attentions_output2input = step_attentions[:, :, :, :input_length]
    step_attentions_output_self = step_attentions[:, :, :, input_length:]
  
    # print(step_attentions_output_self.shape)
    self_att_length = step_attentions_output_self.shape[-1]
    if self_att_length != 0:
        step_attentions_output_self_mean = step_attentions_output_self.sum(axis=-1) / (input_end_index - input_start_index)
        step_attentions_output2input = step_attentions_output2input + step_attentions_output_self_mean.unsqueeze(-1)

    # Remove special tokens
    step_attentions_output2input = step_attentions_output2input[:, :, :, input_start_index:input_end_index]

    # Save the result to CPU as float32 numpy array
    converted_attentions.append(step_attentions_output2input.detach().clone().cpu().to(torch.float32).numpy())

# Print the shapes of the converted tensors
for i, tensor in enumerate(converted_attentions):
    print(f"Shape of output tensor at step {i+1}: {tensor.shape}")  # Should output shape like [28, 28, 1, <length>] keeping only [start_index:end_index]


torch.Size([1, 16, 1, 702])
36
1.0
<class 'tuple'>
torch.Size([1, 16, 1, 702])
Shape of output tensor at step 1: (36, 16, 1, 675)
Shape of output tensor at step 2: (36, 16, 1, 675)
Shape of output tensor at step 3: (36, 16, 1, 675)
Shape of output tensor at step 4: (36, 16, 1, 675)
Shape of output tensor at step 5: (36, 16, 1, 675)
Shape of output tensor at step 6: (36, 16, 1, 675)
Shape of output tensor at step 7: (36, 16, 1, 675)
Shape of output tensor at step 8: (36, 16, 1, 675)
Shape of output tensor at step 9: (36, 16, 1, 675)
Shape of output tensor at step 10: (36, 16, 1, 675)
Shape of output tensor at step 11: (36, 16, 1, 675)
Shape of output tensor at step 12: (36, 16, 1, 675)
Shape of output tensor at step 13: (36, 16, 1, 675)
Shape of output tensor at step 14: (36, 16, 1, 675)
Shape of output tensor at step 15: (36, 16, 1, 675)
Shape of output tensor at step 16: (36, 16, 1, 675)
Shape of output tensor at step 17: (36, 16, 1, 675)
Shape of output tensor at step 18: (36, 16, 1,

In [24]:
def map_token_to_image_position(
    input_ids,
    image_grid_thw,
    merge_size,
    patch_size,
    orig_width,
    orig_height,
    start_pos,
    end_pos
):
    """
    Map the positions of image tokens in input_ids to their original image coordinates.

    Parameters:
        input_ids (torch.Tensor): Tensor of shape (1, seq_len) containing token IDs.
        image_grid_thw (torch.Tensor): Tensor of shape (1, 3) representing image grid info [T, H_patch, W_patch].
        merge_size (int): Merge size in the image processor.
        patch_size (int): Patch size in the image processor.
        orig_width (int): Width of the original image (pixels).
        orig_height (int): Height of the original image (pixels).
        start_pos (int): Starting index of image tokens in input_ids.
        end_pos (int): Ending index of image tokens in input_ids (exclusive).

    Returns:
        list[tuple[float, float, float, float]]: Each image token's original image region (left, top, right, bottom).
    """
    # Get image grid information
    T, H_patch, W_patch = image_grid_thw[0].tolist()
    if T != 1:
        raise ValueError("Only single image (T=1) is supported")

    # Calculate the resized image dimensions
    H = H_patch * patch_size
    W = W_patch * patch_size

    # Compute scaling factors
    scale_w = orig_width / W
    scale_h = orig_height / H

    # Number of patches merged per token
    m = merge_size

    # Total number of image tokens
    num_tokens = (H_patch * W_patch) // (m ** 2)

    # Verify that the number of image tokens matches
    if end_pos - start_pos != num_tokens:
        raise ValueError("Mismatch in number of image tokens")

    # Store the original image coordinates for each token
    positions = []

    # Iterate over each image token
    for i in range(start_pos, end_pos):
        idx = i - start_pos
        # Compute the row and column of the token in the grid
        token_row = idx // (W_patch // m)
        token_col = idx % (W_patch // m)
        # Compute pixel bounds in the resized image
        left = token_col * m * patch_size
        top = token_row * m * patch_size
        right = (token_col + 1) * m * patch_size
        bottom = (token_row + 1) * m * patch_size
        # Map to original image pixel coordinates
        orig_left = left * scale_w
        orig_top = top * scale_h
        orig_right = right * scale_w
        orig_bottom = bottom * scale_h
        # Append to the result list
        positions.append((orig_left, orig_top, orig_right, orig_bottom))

    return positions

import numpy as np
from PIL import Image
import matplotlib.cm as cm
from scipy.ndimage import zoom

def generate_smooth_heatmap(original_image, attention_scores):
    """
    Generate a smoothly interpolated heatmap overlay on the original image.
    Automatically compute H_patch and W_patch based on the length of attention_scores.

    Parameters:
        original_image (PIL.Image.Image): The original image.
        attention_scores (list[float]): List of attention scores.

    Returns:
        PIL.Image.Image: The image with the smooth heatmap overlay.
    """
    # Get the original image dimensions
    orig_width, orig_height = original_image.size
    aspect_ratio = orig_height / orig_width

    # Determine the number of patches from attention_scores length
    num_patches = len(attention_scores)

    # Automatically compute W_patch and H_patch
    W_patch = int(np.round(np.sqrt(num_patches / aspect_ratio)))
    H_patch = int(np.round(W_patch * aspect_ratio))
    grid_size = H_patch * W_patch

    # Handle mismatches between grid_size and attention_scores length
    attention_scores = np.array(attention_scores)
    if grid_size > num_patches:
        # Pad with the mean score if grid_size is larger
        padding_value = np.mean(attention_scores)
        attention_scores = np.pad(attention_scores, (0, grid_size - num_patches),
                                  mode='constant', constant_values=padding_value)
    elif grid_size < num_patches:
        # Truncate if grid_size is smaller
        attention_scores = attention_scores[:grid_size]

    # Reshape into a 2D grid
    attention_grid = attention_scores.reshape((H_patch, W_patch))

    # Bilinearly interpolate to the original image size
    zoom_factor_h = orig_height / H_patch
    zoom_factor_w = orig_width / W_patch
    attention_interp = zoom(attention_grid, (zoom_factor_h, zoom_factor_w), order=1)
    print(attention_interp.shape)
    og_attention_interp = attention_interp.copy()

    # Normalize to [0, 1]
    min_score = attention_interp.min()
    max_score = attention_interp.max()
    if max_score == min_score:
        attention_interp = np.full_like(attention_interp, 0.5)
    else:
        attention_interp = (attention_interp - min_score) / (max_score - min_score)

    # Generate heatmap colors using a colormap
    cmap = cm.get_cmap('jet')
    heatmap_colors = cmap(attention_interp)

    # Create the heatmap image
    heatmap_image = Image.fromarray((heatmap_colors[:, :, :3] * 255).astype(np.uint8))
    heatmap_image = heatmap_image.convert("RGBA")

    # Set heatmap transparency
    alpha = 128
    heatmap_image.putalpha(alpha)

    # Overlay heatmap on the original image
    original_image = original_image.convert("RGBA")
    combined_image = Image.alpha_composite(original_image, heatmap_image)

    return combined_image, og_attention_interp

from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math

def generate_heatmap(original_image, mapping, attention_scores, use_zscore=False):
    """
    Generate a heatmap overlay on the original image based on attention scores.

    Parameters:
        original_image (PIL.Image.Image): The original image object.
        mapping (list[tuple[float, float, float, float]]): List of region coordinates.
        attention_scores (list[float]): List of attention scores.
        use_zscore (bool): Whether to use Z-score normalization (default False).

    Returns:
        PIL.Image.Image: The image with the heatmap overlay.
    """
    # Convert original image to RGBA
    original_image = original_image.convert("RGBA")

    # Normalize attention scores
    if use_zscore:
        # Compute mean and standard deviation
        mean = np.mean(attention_scores)
        std = np.std(attention_scores)
        if std == 0:  # All scores are identical
            normalized_scores = [0.5] * len(attention_scores)
        else:
            # Z-score then sigmoid to map to [0, 1]
            z_scores = [(score - mean) / std for score in attention_scores]
            normalized_scores = [1 / (1 + math.exp(-z)) for z in z_scores]
    else:
        # Min-max normalization
        min_score = min(attention_scores)
        max_score = max(attention_scores)
        if max_score == min_score:
            normalized_scores = [0.5] * len(attention_scores)
        else:
            normalized_scores = [(score - min_score) / (max_score - min_score) for score in attention_scores]

    # Get the colormap
    cmap = cm.get_cmap('jet')
    # Create a transparent heatmap image
    heatmap_image = Image.new("RGBA", original_image.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(heatmap_image)

    # Draw rectangles for each patch
    for (left, top, right, bottom), score in zip(mapping, normalized_scores):
        color = cmap(score)  # Get RGBA color
        color = tuple(int(c * 255) for c in color[:3]) + (128,)  # Set alpha to 128
        left, top, right, bottom = map(round, (left, top, right, bottom))
        draw.rectangle([left, top, right, bottom], fill=color)

    # Composite the heatmap onto the original image
    combined_image = Image.alpha_composite(original_image, heatmap_image)
    return combined_image

from PIL import Image, ImageDraw
import numpy as np
import matplotlib.cm as cm

def generate_smooth_heatmap_mapping(original_image, mapping, attention_scores, sigma=10):
    """
    Generate a smooth heatmap overlay on the original image using specified regions.

    Parameters:
        original_image (PIL.Image.Image): The original image.
        mapping (list[tuple[float, float, float, float]]): List of region coordinates (left, top, right, bottom).
        attention_scores (list[float]): Attention scores corresponding to mapping.
        sigma (float): Gaussian smoothing standard deviation (default 10).

    Returns:
        PIL.Image.Image: The image with the smooth heatmap overlay.
    """
    # Ensure mapping and attention_scores lengths match
    if len(mapping) != len(attention_scores):
        raise ValueError("Length of mapping and attention_scores must match")

    # Get original image dimensions
    orig_width, orig_height = original_image.size

    # Create an empty attention grid
    attention_grid = np.zeros((orig_height, orig_width), dtype=np.float32)

    # Assign scores to the corresponding regions
    for (left, top, right, bottom), score in zip(mapping, attention_scores):
        left, top, right, bottom = map(round, (left, top, right, bottom))
        left = max(0, min(left, orig_width - 1))
        top = max(0, min(top, orig_height - 1))
        right = max(1, min(right, orig_width))
        bottom = max(1, min(bottom, orig_height))
        attention_grid[top:bottom, left:right] = score

    # Apply Gaussian blur for smoothing
    from scipy.ndimage import gaussian_filter
    attention_smooth = gaussian_filter(attention_grid, sigma=sigma)

    # Normalize to [0, 1]
    min_score = attention_smooth.min()
    max_score = attention_smooth.max()
    if max_score == min_score:
        attention_smooth = np.full_like(attention_smooth, 0.5)
    else:
        attention_smooth = (attention_smooth - min_score) / (max_score - min_score)

    # Generate heatmap colors
    cmap = cm.get_cmap('jet')
    heatmap_colors = cmap(attention_smooth)

    # Create heatmap image
    heatmap_image = Image.fromarray((heatmap_colors[:, :, :3] * 255).astype(np.uint8))
    heatmap_image = heatmap_image.convert("RGBA")

    # Set transparency
    alpha = 128
    heatmap_image.putalpha(alpha)

    # Overlay heatmap on the original image
    original_image = original_image.convert("RGBA")
    combined_image = Image.alpha_composite(original_image, heatmap_image)

    return combined_image


import numpy as np

def pad_mask(mask, original_size, padded_size):
    """
    Pad the original mask to match the size of the padded image.

    Parameters:
        mask (np.array): Original mask of shape (H, W).
        original_size (tuple): Original image size (H, W).
        padded_size (tuple): Padded image size (H', W').

    Returns:
        np.array: Padded mask of shape (H', W').
    """
    H, W = original_size
    H_padded, W_padded = padded_size

    # Compute padding amounts (assuming symmetric padding)
    pad_left = (W_padded - W) // 2
    pad_right = W_padded - W - pad_left
    pad_top = (H_padded - H) // 2
    pad_bottom = H_padded - H - pad_top

    # If padded dimensions are smaller, crop the mask
    if H_padded < H:
        mask = mask[:H_padded, :]
    if W_padded < W:
        mask = mask[:, :W_padded]

    # Create the padded mask initialized to zeros
    padded_mask = np.zeros((H_padded, W_padded), dtype=mask.dtype)

    # Copy the original mask into the padded region
    padded_mask[pad_top:pad_top + min(H, H_padded),
                pad_left:pad_left + min(W, W_padded)] = mask

    return padded_mask



In [ ]:
# MACS Algorithm
all_attentions =  [a for a in converted_attentions]
attention_scores =[]
j = 0

for pair in all_attentions:
    res_att_mat = pair.max(axis=1)
    bias_vector = np.ones(pair.shape[-1])
    bias_vector = bias_vector[None, None, :]
    res_att_mat = 0.8*res_att_mat + 0.2*bias_vector 
    joint_att = np.zeros(res_att_mat.shape)
    layers = joint_att.shape[0]
    joint_att[0] = res_att_mat[0]
    for i in np.arange(1, layers):
        joint_att[i] = res_att_mat[i] * joint_att[i-1]  
    attention_scores.append(joint_att)
    j += 1

In [23]:
start,end =3,6
print(f'Response Span: {tokenizer.decode(generated_ids_trimmed[0][start:end])}')
range_step_attentions = attention_scores[start:end]
range_step_attentions_mean = np.array(range_step_attentions).mean(axis=0)
range_step_attentions_mean_last_layer = range_step_attentions_mean[-1,:,:].tolist()[0]
result,og_att  = generate_smooth_heatmap(original_image=image_inputs[0], attention_scores=range_step_attentions_mean_last_layer)
result.show()
result.save('heatmap.png')

Response Span:  a young child
(700, 756)


C:\Users\Kurros\AppData\Local\Temp\ipykernel_21444\3876355359.py:131: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap('jet')


In [ ]:
start,end = 7,10
print(f'Response Span: {tokenizer.decode(generated_ids_trimmed[0][start:end])}')
range_step_attentions = attention_scores[start:end]
range_step_attentions_mean = np.array(range_step_attentions).mean(axis=0)
range_step_attentions_mean_last_layer = range_step_attentions_mean[-1,:,:].tolist()[0]
result,og_att  = generate_smooth_heatmap(original_image=image_inputs[0], attention_scores=range_step_attentions_mean_last_layer)
result.show()
result.save('heatmap.png')

 a dog,
(700, 756)


C:\Users\Kurros\AppData\Local\Temp\ipykernel_21444\3876355359.py:131: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap('jet')


In [25]:
start,end = 35,40
print(f'Response Span: {tokenizer.decode(generated_ids_trimmed[0][start:end])}')
range_step_attentions = attention_scores[start:end]
range_step_attentions_mean = np.array(range_step_attentions).mean(axis=0)
range_step_attentions_mean_last_layer = range_step_attentions_mean[-1,:,:].tolist()[0]
result,og_att  = generate_smooth_heatmap(original_image=image_inputs[0], attention_scores=range_step_attentions_mean_last_layer)
result.show()
result.save('heatmap.png')

Response Span:  red and white jacket,
(700, 756)


C:\Users\Kurros\AppData\Local\Temp\ipykernel_21444\2208080439.py:133: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap('jet')


In [ ]:
# Use this to pin point the response span
for i,t in enumerate(generated_ids_trimmed[0]):
    print(i,tokenizer.decode([t]).replace('\n','\\n'))

0 The
1  image
2  shows
3  a
4  young
5  child
6  and
7  a
8  dog
9 ,
10  likely
11  a
12  Siber
13 ian
14  Hus
15 ky
16 ,
17  sitting
18  together
19  in
20  a
21  grass
22 y
23  outdoor
24  setting
25 .
26  The
27  child
28  is
29  wearing
30  a
31  red
32  cap
33 ,
34  a
35  red
36  and
37  white
38  jacket
39 ,
40  and
41  has
42  a
43  backpack
44  on
45 .
46  The
47  dog
48  is
49  wearing
50  a
51  harness
52  and
53  appears
54  to
55  be
56  licking
57  the
58  child
59 's
60  face
61 .
62  The
63  background
64  includes
65  some
66  trees
67  and
68  a
69  cloudy
70  sky
71 .
72 <|im_end|>
